# # Resume Screener — built by [Kaushik P]
# GitHub: https://github.com/duo311

An end-to-end LLM-powered resume screening system built on **Anthropic Claude**.

### Quick-start
1. Run **Cell 1** to install dependencies
2. Run **Cells 2–6** to set up the project
3. Run **Cell 7** to generate sample data and test the scorer
4. Run **Cell 8** to launch the Streamlit frontend

---

## Cell 1 · Setup & Imports

In [ ]:
# ─── Install dependencies ───────────────────────────────────────────
# Uncomment the line below the first time you run this notebook.
# !pip install anthropic openai PyMuPDF pdfminer.six python-docx streamlit plotly pandas ipywidgets tqdm

import os
import sys
import json
import logging
import time
from pathlib import Path

# ─── Set project root ───────────────────────────────────────────────
PROJECT_ROOT = Path.cwd() / 'resume_screener'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Imports ready')
print(f'   Project root: {PROJECT_ROOT}')

## Cell 2 · Create Folder Structure

In [ ]:
# ─── Create all necessary directories ───────────────────────────────
DIRS = [
    PROJECT_ROOT / 'src',
    PROJECT_ROOT / 'data' / 'resumes',
    PROJECT_ROOT / 'data' / 'output',
    PROJECT_ROOT / 'logs',
]

for d in DIRS:
    d.mkdir(parents=True, exist_ok=True)
    print(f'    {d.relative_to(PROJECT_ROOT.parent)}')

# Touch __init__.py
(PROJECT_ROOT / 'src' / '__init__.py').touch()

print('\n Folder structure ready')

## Cell 3 · ResumeExtractor

In [ ]:
# ─── Verify extractor import and smoke-test ──────────────────────────
from src.extractor import ResumeExtractor

extractor = ResumeExtractor()
print('ResumeExtractor loaded')
print(f'   Supported extensions: {", ".join(sorted(ResumeExtractor.SUPPORTED_EXTENSIONS))}')

# Quick TXT smoke-test
_tmp = PROJECT_ROOT / 'data' / 'resumes' / '_test.txt'
_tmp.write_text('Hello, I am a test resume.')
assert 'Hello' in extractor.extract(_tmp)
_tmp.unlink()
print('   Smoke test: PASS')

## Cell 4 · ResumeScorer (Anthropic / OpenAI)

In [ ]:
# ─── Scorer setup ───────────────────────────────────────────────────
from src.scorer import ResumeScorer, ScoringWeights

# ⚠️  Replace with your actual API key or load from environment:
API_KEY  = os.environ.get('ANTHROPIC_API_KEY', 'YOUR_API_KEY_HERE')
PROVIDER = 'anthropic'  # or 'openai'

weights = ScoringWeights(
    skills=0.40,
    experience=0.30,
    education=0.15,
    cultural_fit=0.15,
)

scorer = ResumeScorer(
    api_key=API_KEY,
    provider=PROVIDER,
    weights=weights,
    max_retries=3,
)

print(f' ResumeScorer ready!')
print(f'   Provider : {scorer.provider}')
print(f'   Model    : {scorer.model}')
print(f'   Weights  : {scorer.weights}')

## Cell 5 · ResumeScreener

In [ ]:
# ─── Screener setup ─────────────────────────────────────────────────
from src.screener import ResumeScreener

screener = ResumeScreener(
    api_key=API_KEY,
    provider=PROVIDER,
    weights=weights,
)

print(' ResumeScreener ready!')
print('   Use screener.screen(file_paths, job_description) for batch processing')
print('   Use screener.screen_texts(texts_dict, job_description) for pre-extracted text')

## Cell 6 · Utility Functions

In [ ]:
# ─── Set up logging and verify utils ────────────────────────────────
from src.utils import (
    setup_logging, save_results_csv, save_results_json,
    load_config, save_config, estimate_tokens,
    format_duration, score_bar,
)

log = setup_logging(log_dir=PROJECT_ROOT / 'logs')
print('✅ Utilities loaded')
print(f'   Log file: {PROJECT_ROOT / "logs" / "screening.log"}')

# Demo
sample_text = 'Python developer with 5 years experience in FastAPI and PostgreSQL.'
print(f'\n   Token estimate demo: "{sample_text[:40]}…"')
print(f'   Estimated tokens : {estimate_tokens(sample_text)}')
print(f'   Score bar (75)   : {score_bar(75)}')

## Cell 7 · Generate Sample Data & Test

In [ ]:
# ─── Sample resumes and job description ─────────────────────────────
RESUME_DIR = PROJECT_ROOT / 'data' / 'resumes'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'output'

JOB_DESCRIPTION = """
Senior Python Developer

We are looking for an experienced Python developer to join our platform team.

Requirements:
- 5+ years of professional Python development
- Strong knowledge of FastAPI or Django REST framework
- Experience with PostgreSQL and Redis
- Proficiency in Docker and Kubernetes
- Familiarity with AWS (EC2, RDS, S3, Lambda)
- Experience with CI/CD pipelines (GitHub Actions, Jenkins)
- Understanding of microservices architecture
- Strong communication skills

Nice to have:
- Open-source contributions
- Experience with ML/AI frameworks (PyTorch, TensorFlow)

Education: Bachelor's or Master's degree in Computer Science or equivalent.
"""

SAMPLE_RESUMES = {
    'alice_chen_strong.txt': """Alice Chen | alice@email.com
Senior Software Engineer — 7 years experience

EXPERIENCE
Lead Python Developer @ TechCorp (2020–Present)
- Architected microservices platform (FastAPI + Kubernetes) handling 2M req/day
- Led team of 6, cut deployment time 60% via GitHub Actions CI/CD
- Tech: Python, FastAPI, PostgreSQL, Redis, Docker, K8s, AWS (EC2, RDS, S3, Lambda)

Senior Python Developer @ DataFlow (2018–2020)
- Built real-time ETL with Apache Kafka + Python
- ML model serving with TensorFlow + FastAPI

SKILLS
Python, FastAPI, Django REST, PostgreSQL, Redis, Docker, Kubernetes,
AWS, GitHub Actions, Jenkins, TensorFlow, PyTorch

EDUCATION
M.Sc. Computer Science — Stanford University (2017)

OPEN SOURCE: Contributor to FastAPI, 3 published PyPI packages
""",
    'bob_martinez_partial.txt': """Bob Martinez | bob.m@gmail.com
Python Developer — 3 years

EXPERIENCE
Python Developer @ Webagency (2022–Present)
- Developed REST APIs with Django REST Framework
- PostgreSQL database design
- Basic Docker containerisation
- Integrated Stripe, Sendgrid APIs

Junior Developer @ CodeShop (2021–2022)
- PHP + Python scripting for data exports
- Basic AWS S3 usage

SKILLS
Python, Django REST Framework, PostgreSQL, Docker (basic), Git, AWS S3

EDUCATION
B.Sc. Information Technology — State University (2021)

Note: No Kubernetes or CI/CD pipeline experience yet.
""",
    'carol_johnson_weak.txt': """Carol Johnson | carol@mail.com
Junior Software Developer — 1 year experience

EXPERIENCE
Junior Developer @ SmallBiz (2023–Present)
- Maintained legacy PHP website
- Wrote Python scripts for data cleanup
- Used Git for version control

Intern @ LocalAgency (2023, 3 months)
- HTML/CSS website updates

SKILLS
Python (beginner), PHP, HTML, CSS, Git, MySQL (basic)

EDUCATION
B.A. Business Administration — Community College (2023)
""",
}

# Write sample resumes to disk
for filename, content in SAMPLE_RESUMES.items():
    path = RESUME_DIR / filename
    path.write_text(content, encoding='utf-8')
    print(f'   Written: {path.name}')

print(f'\n✅ Sample data ready in {RESUME_DIR}')

# ─── Save sample job description ────────────────────────────────────
(PROJECT_ROOT / 'data' / 'job_description.txt').write_text(JOB_DESCRIPTION)
print('   Job description saved')

In [ ]:
# ─── Run batch screening ─────────────────────────────────────────────
# ⚠️  Requires a valid API key set in Cell 4.

if API_KEY == 'YOUR_API_KEY_HERE':
    print('⚠️  Please set API_KEY in Cell 4 before running this cell.')
else:
    from tqdm.notebook import tqdm

    resume_paths = list(RESUME_DIR.glob('*.txt'))
    print(f'Screening {len(resume_paths)} resume(s)…\n')

    pbar = tqdm(total=len(resume_paths), desc='Screening', unit='resume')

    def notebook_progress(current, total, filename):
        if current > 0:
            pbar.update(1)
            pbar.set_postfix_str(filename)

    session = screener.screen(
        file_paths=resume_paths,
        job_description=JOB_DESCRIPTION,
        output_dir=OUTPUT_DIR,
        progress_callback=notebook_progress,
    )
    pbar.close()

    # ─── Print ranked results ────────────────────────────────────────
    print('\n' + '═' * 60)
    print('  SCREENING RESULTS')
    print('═' * 60)
    print(session.summary())
    print('─' * 60)

    from src.utils import score_bar, recommendation_color

    for rank, r in enumerate(session.ranked, 1):
        print(f'\n#{rank}  {r.filename}')
        print(f'    Score : {score_bar(r.overall_score)}')
        print(f'    Rec   : {r.recommendation}')
        print(f'    Exp   : {r.years_of_experience} yrs  |  Edu: {r.education_level}')
        if r.matched_skills:
            print(f'    ✅ Matched : {", ".join(r.matched_skills[:5])}')
        if r.missing_skills:
            print(f'    ❌ Missing : {", ".join(r.missing_skills[:5])}')
        print(f'    📝 {r.summary[:120]}…')

    print('\n' + '═' * 60)
    print(f'Results saved to {OUTPUT_DIR}')

In [ ]:
# ─── Quick visualisation in notebook ────────────────────────────────
try:
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    if 'session' in dir() and session.results:
        ranked = session.ranked
        names  = [r.filename.replace('.txt', '') for r in ranked]
        scores = [r.overall_score for r in ranked]
        colour_map = {'Strong Yes': '#10b981', 'Yes': '#3b82f6', 'Maybe': '#f59e0b', 'No': '#ef4444'}
        colours = [colour_map.get(r.recommendation, 'grey') for r in ranked]

        fig, ax = plt.subplots(figsize=(10, max(3, len(ranked) * 1.5)))
        bars = ax.barh(names, scores, color=colours, edgecolor='white', height=0.6)
        ax.set_xlim(0, 110)
        ax.axvline(80, color='#10b981', linestyle='--', alpha=.6, label='Strong Yes (80)')
        ax.axvline(65, color='#3b82f6', linestyle='--', alpha=.6, label='Yes (65)')
        ax.axvline(45, color='#f59e0b', linestyle='--', alpha=.6, label='Maybe (45)')
        for bar, score in zip(bars, scores):
            ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                    f'{score:.1f}', va='center', fontsize=11, fontweight='bold')
        ax.set_xlabel('Score (0–100)', fontsize=12)
        ax.set_title('Resume Screening Results', fontsize=14, fontweight='bold', pad=12)
        ax.legend(loc='lower right', fontsize=9)
        ax.invert_yaxis()
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'score_chart.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('Chart saved to', OUTPUT_DIR / 'score_chart.png')
    else:
        print('Run the screening cell first to see the chart.')
except ImportError:
    print('Install matplotlib to see the chart: pip install matplotlib')

## Cell 8 · Launch Streamlit Frontend

In [ ]:
# ─── Launch the Streamlit app ────────────────────────────────────────
# This opens the app in a new browser tab.
# If running on a remote server / Colab, see the URL printed below.

import subprocess, threading, time

APP_PATH = PROJECT_ROOT / 'app.py'
PORT = 8501

def run_streamlit():
    subprocess.run(
        ['streamlit', 'run', str(APP_PATH),
         '--server.port', str(PORT),
         '--server.headless', 'true'],
        check=False,
    )

# Start in background thread so notebook stays responsive
t = threading.Thread(target=run_streamlit, daemon=True)
t.start()
time.sleep(3)  # Give Streamlit a moment to start

print(f'🚀 Streamlit app running!')
print(f'   Local URL : http://localhost:{PORT}')
print()
print('   If running in JupyterHub or a remote environment,')
print(f'   forward port {PORT} or use the public URL printed by Streamlit above.')
print()
print('   To stop: restart the Jupyter kernel.')

---

## 📖 Instructions

### 1 — Install dependencies
```bash
pip install anthropic openai PyMuPDF pdfminer.six python-docx \
            streamlit plotly pandas ipywidgets tqdm matplotlib
```
Or run:
```bash
pip install -r resume_screener/requirements.txt
```

### 2 — Set up your API key
**Option A — environment variable (recommended):**
```bash
export ANTHROPIC_API_KEY=sk-ant-...
```
**Option B — paste directly in Cell 4:** set `API_KEY = 'sk-ant-...'`

**Option C — Streamlit UI:** enter the key in the sidebar at runtime.

### 3 — Run the Streamlit app
Either run **Cell 8** in this notebook, or from the terminal:
```bash
cd resume_screener
streamlit run app.py
```

### 4 — Using the system

**From the notebook:**
1. Run Cells 1–6 in order to initialise everything.
2. Run Cell 7 to generate sample resumes and test scoring.
3. Place your own resumes in `data/resumes/` and call `screener.screen()`.

**From the Streamlit UI:**
1. Enter your API key in the sidebar.
2. Paste your job description.
3. Upload resume files (PDF, DOCX, or TXT).
4. Adjust scoring weights to match your priorities.
5. Click **Start Screening**.
6. View ranked results, radar charts, and download CSV/JSON exports.

### 5 — Output files
| File | Description |
|------|-------------|
| `data/output/screening_results.csv` | Ranked scores for all candidates |
| `data/output/screening_results.json` | Full structured analysis |
| `data/output/score_chart.png` | Bar chart (generated in Cell 7) |
| `logs/screening.log` | Detailed processing log |

### 6 — Switching to OpenAI
In Cell 4, change:
```python
API_KEY  = 'sk-...'          # OpenAI key
PROVIDER = 'openai'
```
Or in the Streamlit sidebar, select **OpenAI (GPT)**.

### 7 — Cost estimation
Before scoring, call:
```python
scorer.estimate_cost(resume_text, job_description)
# → {'estimated_tokens': 1250, 'estimated_cost_usd': 0.0038}
```

### Tips
- For best results, include **required skills** clearly in your job description.
- Increase the **Skills** weight for highly technical roles.
- Increase the **Experience** weight for senior positions.
- Scores are AI-generated estimates — always apply human judgement.
